In [57]:
import polars as pl
import pandas as pd

# File Paths
vcf_path = "toy_trimmed.vcf" 
results_path = "matched_snps_output.vcf" 

# === Load VCF ===
vcf = pl.read_csv(vcf_path, separator="\t", has_header=True, comment_prefix="##")
vcf = vcf.rename({"#CHROM": "CHROM"})

# ALT genotypes of interest
alt_genos = {"0/1", "1/0", "1/1"}

# Load predictions
predictions = pd.read_csv(results_path, sep="\t")
tissues = ["RNA-seq: Putamen", "RNA-seq: Caudate"]

# Initialize: {tissue: list of rows}
tissue_long_rows = {t: [] for t in tissues}

print(f"VCF shape: {vcf.shape}")
print(f"Predictions shape: {predictions.shape}")

VCF shape: (30, 13)
Predictions shape: (300, 5)


I tested the same analysis with less data such as 10 rows in the VCF file and 50 rows in the predictions but the problem was that there ware not more than one SNP affecting the same gene so there was not a lot to see

In [58]:
meta_cols = {"CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"}
sample_names = [col for col in vcf.columns if col not in meta_cols]
print(f"Samples found: {sample_names}")


Samples found: ['gwa1.100_gwa1.100', 'gwa1.1012_gwa1.1012', 'gwa1.1016_gwa1.1016', 'gwa1.101_gwa1.101']


Now printing  how many SNPS have an ALT alelle for each Sample and a list of these SNPS

In [59]:
sample_snps = {}

for sample_name in sample_names:
    snps = vcf.filter(pl.col(sample_name).is_in(alt_genos)).select("ID").to_series().to_list()
    snps = set(str(s).strip() for s in snps)
    sample_snps[sample_name] = snps
    print(f"\nSample: {sample_name}")
    print(f"Number of ALT SNPs: {len(snps)}")
    print(f"ALT SNPs: {sorted(snps)}")




Sample: gwa1.100_gwa1.100
Number of ALT SNPs: 11
ALT SNPs: ['18_10421654_A_G', '18_11986656_C_A', '18_11986706_C_T', '18_11993138_C_G', '18_20436874_C_T', '18_314443_A_G', '18_32956524_G_A', '18_46468251_T_C', '18_55110898_G_C', '18_56517508_T_C', '18_74103092_T_C']

Sample: gwa1.1012_gwa1.1012
Number of ALT SNPs: 10
ALT SNPs: ['18_10421654_A_G', '18_11979422_G_A', '18_11986656_C_A', '18_11986706_C_T', '18_20436874_C_T', '18_314443_A_G', '18_47494393_G_A', '18_56517508_T_C', '18_74103092_T_C', '18_9806921_A_G']

Sample: gwa1.1016_gwa1.1016
Number of ALT SNPs: 10
ALT SNPs: ['18_10421654_A_G', '18_20436874_C_T', '18_314443_A_G', '18_39763789_C_G', '18_47494393_G_A', '18_55816329_G_A', '18_56517508_T_C', '18_679660_C_G', '18_685511_T_C', '18_74103092_T_C']

Sample: gwa1.101_gwa1.101
Number of ALT SNPs: 14
ALT SNPs: ['18_11979422_G_A', '18_11986656_C_A', '18_11986706_C_T', '18_11993138_C_G', '18_20436874_C_T', '18_3075778_A_C', '18_314443_A_G', '18_32956524_G_A', '18_46468251_T_C', '18_47

Now just for the first sample i am printing all the matched predictions for both tissues 
Test can be done for the other 3 samples by changing the sample here:
sample_to_check = sample_names[0] 

In [60]:
sample_to_check = sample_names[0] 
snps = sample_snps[sample_to_check]

sample_preds = predictions[predictions["snp"].isin(snps)]
print(f"\nSample: {sample_to_check}")
print(f"Matching prediction rows: {len(sample_preds)}")
print(sample_preds)



Sample: gwa1.100_gwa1.100
Matching prediction rows: 118
                 snp             gene alt_allele            tissue    logSED
20   18_74103092_T_C  ENSG00000101493          C  RNA-seq: Putamen -0.000076
21   18_74103092_T_C  ENSG00000101493          C  RNA-seq: Caudate  0.000466
22   18_74103092_T_C  ENSG00000263982          C  RNA-seq: Putamen -0.000215
23   18_74103092_T_C  ENSG00000263982          C  RNA-seq: Caudate -0.000433
24   18_10421654_A_G  ENSG00000287065          G  RNA-seq: Putamen  0.000006
..               ...              ...        ...               ...       ...
273  18_11986656_C_A  ENSG00000266955          A  RNA-seq: Caudate  0.000504
274  18_11986656_C_A  ENSG00000267480          A  RNA-seq: Putamen  0.000478
275  18_11986656_C_A  ENSG00000267480          A  RNA-seq: Caudate  0.000407
276  18_11986656_C_A  ENSG00000267722          A  RNA-seq: Putamen  0.000020
277  18_11986656_C_A  ENSG00000267722          A  RNA-seq: Caudate  0.000025

[118 rows x 5 colu

Printing the final logSED sum for the tissue Putamen for the same sample only

In [ ]:
#for tissue in tissues:
tissue_preds = sample_preds[sample_preds["tissue"] == tissues[0]]
print(f"\nTissue: {tissues[0]}")
print(f"Tissue predictions: {len(tissue_preds)}")
    
   

grouped = tissue_preds.groupby("gene")["logSED"].sum().reset_index()
print(f"Aggregated expression vector for {sample_to_check} - {tissues[0]}")
print(grouped)



Tissue: RNA-seq: Putamen
Tissue predictions: 59
Aggregated expression vector for gwa1.100_gwa1.100 - RNA-seq: Putamen
               gene    logSED
0   ENSG00000074657 -0.000266
1   ENSG00000079134 -0.000095
2   ENSG00000101493 -0.000076
3   ENSG00000101665  0.001389
4   ENSG00000101773 -0.000465
5   ENSG00000119547 -0.000125
6   ENSG00000134030 -0.000452
7   ENSG00000141401 -0.002920
8   ENSG00000153391  0.001453
9   ENSG00000154856  0.001127
10  ENSG00000154889 -0.000109
11  ENSG00000158270  0.001146
12  ENSG00000172175  0.000063
13  ENSG00000172466 -0.000143
14  ENSG00000177511 -0.000049
15  ENSG00000186496 -0.004745
16  ENSG00000186814 -0.001318
17  ENSG00000260913  0.003387
18  ENSG00000263884 -0.001205
19  ENSG00000263982 -0.000215
20  ENSG00000264269 -0.000062
21  ENSG00000264514  0.000620
22  ENSG00000264876 -0.000009
23  ENSG00000265943 -0.000533
24  ENSG00000266102  0.000014
25  ENSG00000266850 -0.000003
26  ENSG00000266905  0.000209
27  ENSG00000266955  0.001471
28  ENSG000

For the same Sample AND tissue again :
Printing for each Gene 
A list of the snps that contributed to the sum plus the logSED values for validation of the sum
Prints only if there is 2 or more snps afecting

In [62]:
# Print detailed contribution for each gene
for gene, group in tissue_preds.groupby("gene"):
    snp_ids = group["snp"].tolist()
    logsed_vals = group["logSED"].tolist()

    if len(logsed_vals) <= 1:
        continue  # Skip genes with only one logSED value

    total = sum(logsed_vals)

    print(f"\nGene: {gene}")
    print(f"  SNPs used:     {', '.join(snp_ids)}")
    print(f"  logSED values: {', '.join([str(round(v, 7)) for v in logsed_vals])}")
    print(f"  Sum:           {round(total, 7)}")



Gene: ENSG00000141401
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: 0.0001897, -0.001048, -0.002062
  Sum:           -0.0029203

Gene: ENSG00000154889
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: -2.69e-05, 0.0003166, -0.0003984
  Sum:           -0.0001087

Gene: ENSG00000266955
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: 0.0006475, 0.00016, 0.0006638
  Sum:           0.0014713

Gene: ENSG00000267079
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: 0.0002221, 2.54e-05, 0.000554
  Sum:           0.0008015

Gene: ENSG00000267480
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: -0.00049, -0.000202, 0.0004785
  Sum:           -0.0002135

Gene: ENSG00000267722
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: 0.0001315, 1.13e-05, 1.97e-05
  Sum:           0.0001625

Gene: 

Printing the final logSED sum for the tissue Caudate for the same sample only

In [ ]:
tissue_preds = sample_preds[sample_preds["tissue"] == tissues[1]]
print(f"\nTissue: {tissues[1]}")
print(f"Tissue predictions: {len(tissue_preds)}")
    
   

grouped = tissue_preds.groupby("gene")["logSED"].sum().reset_index()
print(f"Aggregated expression vector for {sample_to_check} - {tissues[1]}")
print(grouped)



Tissue: RNA-seq: Caudate
Tissue predictions: 59
Aggregated expression vector for gwa1.100_gwa1.100 - RNA-seq: Putamen
               gene    logSED
0   ENSG00000074657 -0.000943
1   ENSG00000079134  0.000004
2   ENSG00000101493  0.000466
3   ENSG00000101665  0.001039
4   ENSG00000101773 -0.000444
5   ENSG00000119547  0.000393
6   ENSG00000134030 -0.000215
7   ENSG00000141401 -0.000505
8   ENSG00000153391  0.001415
9   ENSG00000154856  0.000999
10  ENSG00000154889 -0.000211
11  ENSG00000158270  0.001801
12  ENSG00000172175  0.000024
13  ENSG00000172466  0.001030
14  ENSG00000177511  0.000389
15  ENSG00000186496 -0.005420
16  ENSG00000186814 -0.000949
17  ENSG00000260913 -0.001863
18  ENSG00000263884 -0.000822
19  ENSG00000263982 -0.000433
20  ENSG00000264269 -0.000199
21  ENSG00000264514  0.000705
22  ENSG00000264876 -0.000057
23  ENSG00000265943 -0.000908
24  ENSG00000266102 -0.000011
25  ENSG00000266850  0.000073
26  ENSG00000266905 -0.000129
27  ENSG00000266955  0.000611
28  ENSG000

For the same Sample AND tissue again:
Printing for each Gene 
A list of the snps that contributed to the sum plus the logSED values for validation of the sum
Prints only if there is 2 or more snps afecting

In [64]:
# Print detailed contribution for each gene
for gene, group in tissue_preds.groupby("gene"):
    snp_ids = group["snp"].tolist()
    logsed_vals = group["logSED"].tolist()

    if len(logsed_vals) <= 1:
        continue  # Skip genes with only one logSED value

    total = sum(logsed_vals)

    print(f"\nGene: {gene}")
    print(f"  SNPs used:     {', '.join(snp_ids)}")
    print(f"  logSED values: {', '.join([str(round(v, 7)) for v in logsed_vals])}")
    print(f"  Sum:           {round(total, 7)}")


Gene: ENSG00000141401
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: 0.0004306, -0.0003166, -0.000619
  Sum:           -0.000505

Gene: ENSG00000154889
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: -0.000102, 0.0002335, -0.0003424
  Sum:           -0.0002109

Gene: ENSG00000266955
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: -7.4e-06, 0.0001144, 0.0005045
  Sum:           0.0006115

Gene: ENSG00000267079
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: 0.0004623, 4.92e-05, 0.0003564
  Sum:           0.0008679

Gene: ENSG00000267480
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: -7.86e-05, -0.0001026, 0.0004067
  Sum:           0.0002255

Gene: ENSG00000267722
  SNPs used:     18_11993138_C_G, 18_11986706_C_T, 18_11986656_C_A
  logSED values: 4.64e-05, 2.6e-06, 2.5e-05
  Sum:           7.41e-05

Gene: E